In [1]:
import json
from pathlib import Path
from collections import defaultdict

%load_ext autoreload
%autoreload 2

from utgen.raggraph.walker import build_graph_from_directory
from utgen.raggraph.utils import get_node_context
from utgen.test_generation_crew.crew import TestGenerationCrew
from utgen.validation import validate_individual_test, save_and_clean_tests

In [ ]:
# Params
SOURCE_CODE_DIR = "../src"
TESTS_OUTPUT_DIR = "../tests"
SAVE_GRAPH = ""
SAVE_GRAPH_PATH = "../graph.json"

In [ ]:
g = build_graph_from_directory(code_path=SOURCE_CODE_DIR, save_graph_path=SAVE_GRAPH_PATH)

In [ ]:
# Define defaultdict of dicts
tests_results = defaultdict(dict)

# TODO: afegir guardrails que falten
test_generator = TestGenerationCrew(guardrail_max_retries=3, verbose=False)

for node_id, data in list(g.nodes(data=True)):
    if data["type"] in ["function", "method"]:
        print(f"Generating tests for node: {node_id}")
        
        try:
            # Get context
            context = get_node_context(g=g, node_id=node_id)
            inputs = {"graph_context": context}

            # Generate tests
            response = test_generator.crew().kickoff(inputs=inputs)

            # Convert string to dictionary
            response_dict = json.loads(response.raw)

            # Store results
            p = Path(data["file"])
            new_filename = f"test_{p.stem}{p.suffix}"
            save_path = (p.parent / new_filename).as_posix()
            tests_results[save_path][node_id] = response_dict['tests']

        except Exception as e:
            # This catches guardrail retries exceeded, JSON parsing errors, etc.
            print(f"⚠️ Failed to generate tests for {node_id} after max retries.")
            print(f"Error: {e}")
            # Use 'continue' to skip the rest of this iteration and move to the next node
            continue

Generating tests for node: utgen/logger.py::function::disable_dependency_loggers
Generating tests for node: utgen/logger.py::function::setup_logger
Generating tests for node: utgen/validation.py::function::validate_individual_test
Generating tests for node: utgen/validation.py::function::save_and_clean_tests
Generating tests for node: utgen/raggraph/utils.py::function::get_node_context



2026-03-16 22:21:26.335 | WARNING  | utgen.test_generation_crew.guardrails:validate_tests_schema:53 - Guardrail `validate_tests_schema` triggered: invalid JSON format


Generating tests for node: utgen/raggraph/utils.py::function::get_source_segment
Generating tests for node: utgen/raggraph/utils.py::function::canonical_id
Generating tests for node: utgen/raggraph/utils.py::function::normalize_signature



2026-03-16 22:24:37.523 | WARNING  | utgen.test_generation_crew.guardrails:validate_tests_schema:94 - Guardrail `validate_tests_schema` triggered: invalid test entries.


Generating tests for node: utgen/raggraph/walker.py::function::iter_python_files
Generating tests for node: utgen/raggraph/walker.py::function::build_graph_from_directory
Generating tests for node: utgen/test_generation_crew/guardrails.py::function::validate_tests_schema


In [ ]:
for save_path, nodes in tests_results.items():
    tests_que_han_passat = []
    for node_id, tests in nodes.items():
        # Validem el test generat
        base_import = 'from ' + node_id.split("::")[0][:-3].replace("/", ".") + ' import ' + node_id.split("::")[-1].split(".")[0]
        for name, values in tests.items():
            imports, code = values['imports'], values['code']
            # Afegim el import base per si no està present
            if base_import not in imports:
                print(f"⚠️ Import base `{base_import}` no present en el test `{name}`, afegint-lo...")
                imports.append(base_import)
            imports = "\n".join(imports)

            if validate_individual_test(import_code=imports, test_code=code):
                print(f"✅ Test `{name}` acceptat")
                tests_que_han_passat.append((imports, code))
            else:
                print(f"❌ Test `{name}` rebutjat")

    # Guardem i deixem el fitxer perfecte
    print(f"\nGuardant tests per `{save_path}`...")
    save_and_clean_tests(valid_tests=tests_que_han_passat, destination=f"{TESTS_OUTPUT_DIR}/{save_path}")

# TODO: run pytest coverage

⚠️ Import base `from utgen.logger import disable_dependency_loggers` no present en el test `test_disable_dependency_loggers_with_valid_input`, afegint-lo...
❌ Test `test_disable_dependency_loggers_with_valid_input` rebutjat
⚠️ Import base `from utgen.logger import disable_dependency_loggers` no present en el test `test_disable_dependency_loggers_with_empty_list`, afegint-lo...
✅ Test `test_disable_dependency_loggers_with_empty_list` acceptat
⚠️ Import base `from utgen.logger import disable_dependency_loggers` no present en el test `test_disable_dependency_loggers_with_none_input`, afegint-lo...
✅ Test `test_disable_dependency_loggers_with_none_input` acceptat
⚠️ Import base `from utgen.logger import disable_dependency_loggers` no present en el test `test_disable_dependency_loggers_with_non_string_items`, afegint-lo...
❌ Test `test_disable_dependency_loggers_with_non_string_items` rebutjat
⚠️ Import base `from utgen.logger import setup_logger` no present en el test `test_setup_logger_wi